In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv(r"C:\Users\BHUPATHI NADAR\OneDrive\Desktop\Main_project\MLSC-Task\Data\data.csv")

In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\BHUPATHI NADAR\OneDrive\Desktop\Main_project\MLSC-Task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1253.80it/s]


In [5]:
response_embeddings = embedding_model.encode(
    df["response"].fillna("").tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 59/59 [00:09<00:00,  6.05it/s]


In [7]:
def documents_to_text(documents):
    if isinstance(documents, (list, tuple, np.ndarray)):
        return " ".join(map(str, documents))
    return str(documents)

In [8]:
documents_text = df["documents"].apply(documents_to_text)

In [9]:
document_embeddings = embedding_model.encode(
    documents_text.tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 59/59 [01:05<00:00,  1.11s/it]


In [11]:
question_embeddings = embedding_model.encode(
    df["question"].fillna("").tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 59/59 [00:05<00:00, 11.62it/s]


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

doc_response_similarity = cosine_similarity(
    document_embeddings,
    response_embeddings
).diagonal()

question_response_similarity = cosine_similarity(
    question_embeddings,
    response_embeddings
).diagonal()

question_document_similarity = cosine_similarity(
    question_embeddings,
    document_embeddings
).diagonal()

In [16]:
df["question_embeddings"]=list(question_embeddings)

df["document_embeddings"]=list(document_embeddings)

df["response_embeddings"]=list(response_embeddings)

df["doc_response_similarity"] = doc_response_similarity

df["question_response_similarity"] = question_response_similarity

df["question_document_similarity"] = question_document_similarity

In [17]:
df.head()

,question,documents,response,label,doc_response_similarity,question_response_similarity,question_document_similarity,question_embeddings,document_embeddings,response_embeddings
0,In what school district is Governor John R. Ro...,['Governor John R. Rogers High School is a hig...,"Governor John R. Rogers High School, named aft...",GROUNDED,0.734998,0.923932,0.710141,"[-0.08299309, 0.07601401, 0.011534999, 0.05428...","[-0.057945106, -0.0018210701, 0.061407734, 0.0...","[-0.08348218, 0.076349735, 0.00020649731, 0.03..."
1,whiich Australian racing driver won the 44-lap...,"['Colin Fleming (born April 21, 1984 in San Di...",Daniel Ricciardo won the 44-lap race for the R...,GROUNDED,0.504248,0.743390,0.466463,"[0.025837678, 0.045963217, -0.07016523, 0.0584...","[-0.052254546, -0.09303288, -0.060229503, 4.98...","[0.036320187, 0.016421901, -0.059343863, 0.040..."
2,What star of Parks and Recreation appeared in ...,['Pioneer Park is a 44-acre (109-ha) city park...,"Nick Offerman, who played the role of Ron Swan...",GROUNDED,0.245878,0.616227,0.371195,"[0.0035456799, -0.006405758, -0.001759537, -0....","[0.0032229996, -0.03810048, 0.027101034, -0.02...","[-0.10564802, -0.06891022, -0.014734206, -0.06..."
3,Which genus of flowering plant is found in an ...,['Actaea arizonica is a species of flowering p...,Crocosmia is found in an environment further s...,GROUNDED,0.462568,0.617127,0.690663,"[0.012242848, 0.022732468, -0.058080446, -0.00...","[0.017341288, -0.029099816, -0.13269252, 0.012...","[0.07983518, -0.006273954, -0.042168286, 0.037..."
4,In what year did the man who shot the Chris St...,['Dennis Bruce Allen (7 November 1951 – 13 Apr...,"Dennis Bruce Allen, the man who shot Chris Sto...",GROUNDED,0.485141,0.804504,0.409791,"[0.009221579, 0.06326537, -0.09361036, 0.00649...","[0.023946796, -0.07257524, -0.081174746, -0.08...","[0.00290469, 0.0663615, -0.11296855, -0.011436..."


In [19]:
df.to_csv(r"C:\Users\BHUPATHI NADAR\OneDrive\Desktop\Main_project\MLSC-Task\Data\semantic_data.csv", index=False)